# Are.na Channel → Archive Webpage
# with CSS Layout Options

This notebook does two things:

1. Pulls blocks from a public Are.na channel
2. Teaches you **flexbox** and **grid** so you can choose and customize a layout for your archive page

You'll generate your page, then pick from four layout options — or combine them.

No API key needed for public channels.

---
## Step 1: Set your channel slug

Go to your Are.na channel. The slug is the last part of the URL.

If your channel URL is:
```
https://www.are.na/alexa-ann-bonomo/relics-uikeqpbrexa
```
Then your slug is `relics-uikeqpbrexa`.

In [ ]:
CHANNEL_SLUG = 'relics-uikeqpbrexa'
PAGE_TITLE = 'relics'

---
## Step 2: Fetch blocks from Are.na

Each block has a `class` that tells us its content type: `Image`, `Link`, `Text`, `Media`, `Attachment`, or `Channel`.

Other fields we can use:
- `title` or `generated_title`
- `description` — often blank
- `source.url` — where a link came from (might be dead)
- `image.display.url` — hosted image (might be gone)
- `connected_at` — when someone added it
- `user.username` — who added it
- `content` — the text body for Text blocks

In [ ]:
import requests
import json

url = f'https://api.are.na/v2/channels/{CHANNEL_SLUG}?per=100'
response = requests.get(url)

if response.status_code != 200:
    print(f'Error: API returned {response.status_code}')
    print('Make sure the channel is public and the slug is correct.')
else:
    data = response.json()
    blocks = [b for b in data.get('contents', []) if b.get('class') != 'Channel']
    channel_title = data.get('title', 'untitled')
    channel_user = data.get('user', {}).get('username', '')
    channel_length = data.get('length', 0)
    channel_updated = data.get('updated_at', '')[:10]
    print(f'Channel: {channel_title}')
    print(f'By: {channel_user}')
    print(f'Blocks: {len(blocks)}')
    print(f'Updated: {channel_updated}')
    print()
    for b in blocks[:5]:
        print(f'  [{b["class"]}] {b.get("title", b.get("generated_title", "(no title)"))}')
    if len(blocks) > 5:
        print(f'  ... and {len(blocks) - 5} more')

---
## Step 3: Inspect a single block

Look at the raw data for one block. This is what you have to work with.

In [ ]:
if blocks:
    print(json.dumps(blocks[0], indent=2, default=str))
else:
    print('No blocks to inspect.')

---
---
# Part 2: Learning Flexbox and Grid

Before we generate the final page, let's learn the two CSS layout systems you'll use to arrange your blocks.

Run the helper cell below, then work through each section.

In [ ]:
from IPython.display import display, HTML
import html as html_module

def preview(code, height=500):
    escaped = html_module.escape(code)
    iframe = f"""
    <iframe srcdoc="{escaped}" style="width:100%; height:{height}px; border:1px solid #999;
    background:#fff;" sandbox="allow-scripts"></iframe>
    """
    display(HTML(iframe))

print('preview helper loaded')

---
### Flexbox

Flexbox is **one-dimensional** — it lays items out along a single row or column.

| Property | What it does |
|---|---|
| `display: flex` | Makes a container a flex container |
| `flex-direction` | `row` (default) or `column` |
| `justify-content` | Distributes items along the main axis |
| `align-items` | Aligns items along the cross axis |
| `flex-wrap: wrap` | Lets items flow to the next line |
| `gap` | Space between items |
| `flex: grow shrink basis` | On children — how they fill space |

#### Flex rows and justify-content

Three versions of the same row of blocks, with different `justify-content` values.

In [ ]:
flex_justify = """
<!DOCTYPE html>
<html>
<head>
<style>
  body { margin: 16px; font-family: 'Times New Roman', Times, serif; font-size: 12px; }
  h3 { font-size: 11px; margin: 0 0 6px; color: #666; }

  .row {
    display: flex;
    gap: 8px;
    margin-bottom: 20px;
  }

  .block {
    border: 1px solid #999;
    padding: 10px;
    min-width: 100px;
  }
  .block .title { font-weight: bold; margin-bottom: 2px; }
  .block .type { color: #888; font-size: 10px; }
</style>
</head>
<body>

  <h3>justify-content: flex-start (default)</h3>
  <div class="row" style="justify-content: flex-start;">
    <div class="block"><div class="title">Untitled</div><div class="type">Image</div></div>
    <div class="block"><div class="title">Parking lot</div><div class="type">Link</div></div>
    <div class="block"><div class="title">notes.txt</div><div class="type">Text</div></div>
  </div>

  <h3>justify-content: space-between</h3>
  <div class="row" style="justify-content: space-between;">
    <div class="block"><div class="title">Untitled</div><div class="type">Image</div></div>
    <div class="block"><div class="title">Parking lot</div><div class="type">Link</div></div>
    <div class="block"><div class="title">notes.txt</div><div class="type">Text</div></div>
  </div>

  <h3>justify-content: center</h3>
  <div class="row" style="justify-content: center;">
    <div class="block"><div class="title">Untitled</div><div class="type">Image</div></div>
    <div class="block"><div class="title">Parking lot</div><div class="type">Link</div></div>
    <div class="block"><div class="title">notes.txt</div><div class="type">Text</div></div>
  </div>

</body>
</html>
"""
preview(flex_justify, 310)

**Try it:** Change `flex-direction: row` to `flex-direction: column` on one of the `.row` divs and re-run.

#### Flex wrap and the `flex` property

`flex-wrap: wrap` lets items flow to the next line. The `flex` shorthand on children controls how they grow.

In [ ]:
flex_wrap = """
<!DOCTYPE html>
<html>
<head>
<style>
  body { margin: 16px; font-family: 'Times New Roman', Times, serif; font-size: 12px; }
  h3 { font-size: 11px; margin: 0 0 6px; color: #666; }
  p.note { font-size: 11px; color: #888; margin: 0 0 10px; }

  .wrap-row {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
  }

  .card {
    flex: 1 1 140px;
    /* flex-grow: 1 — cards grow to fill leftover space   */
    /* flex-shrink: 1 — cards can shrink if needed         */
    /* flex-basis: 140px — each card starts at 140px wide  */
    border: 1px solid #999;
    padding: 10px;
  }
  .card .title { font-weight: bold; margin-bottom: 2px; }
  .card .type { color: #888; font-size: 10px; }
  .card .date { color: #aaa; font-size: 10px; }
</style>
</head>
<body>
  <h3>flex-wrap: wrap with flex: 1 1 140px</h3>
  <p class="note">Cards grow to fill space. Resize your browser to see them reflow.</p>
  <div class="wrap-row">
    <div class="card"><div class="title">Untitled</div><div class="type">Image</div><div class="date">2025-01-12</div></div>
    <div class="card"><div class="title">Parking lot</div><div class="type">Link</div><div class="date">2025-01-14</div></div>
    <div class="card"><div class="title">notes.txt</div><div class="type">Text</div><div class="date">2025-02-03</div></div>
    <div class="card"><div class="title">Untitled</div><div class="type">Image</div><div class="date">2025-02-10</div></div>
    <div class="card"><div class="title">dead link</div><div class="type">Link</div><div class="date">2025-03-01</div></div>
    <div class="card"><div class="title">Untitled</div><div class="type">Attachment</div><div class="date">2025-03-08</div></div>
  </div>
</body>
</html>
"""
preview(flex_wrap, 230)

**Try it:** Change `flex: 1 1 140px` to `flex: 0 0 200px`. Cards no longer grow — they stay fixed at 200px.

#### Flexbox for navigation

Flexbox is good for headers, nav bars, and filter rows — anywhere you need items spread across a single line.

In [ ]:
flex_nav = """
<!DOCTYPE html>
<html>
<head>
<style>
  body { margin: 0; font-family: 'Times New Roman', Times, serif; font-size: 12px; }

  .header {
    display: flex;
    justify-content: space-between;
    align-items: baseline;
    padding: 14px 16px;
    border-bottom: 1px solid black;
  }
  .header h1 { font-size: 14px; margin: 0; }
  .header nav { display: flex; gap: 14px; }
  .header nav a { color: black; text-decoration: none; font-size: 11px; }
  .header nav a:hover { text-decoration: underline; }

  .filters {
    display: flex;
    flex-wrap: wrap;
    gap: 6px;
    padding: 10px 16px;
    border-bottom: 1px solid #ccc;
  }
  .filters button {
    font-family: inherit;
    font-size: 10px;
    padding: 3px 8px;
    border: 1px solid #999;
    background: white;
    cursor: pointer;
  }
  .filters button.on { background: black; color: white; }

  .footer {
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 2px;
    padding: 16px;
    font-size: 10px;
    color: #999;
    border-top: 1px solid #ccc;
    margin-top: 16px;
  }
</style>
</head>
<body>
  <div class="header">
    <h1>channel title</h1>
    <nav>
      <a href="#">index</a>
      <a href="#">about</a>
      <a href="#">are.na</a>
    </nav>
  </div>
  <div class="filters">
    <button class="on">all</button>
    <button>Image</button>
    <button>Link</button>
    <button>Text</button>
    <button>Media</button>
    <button>Attachment</button>
  </div>
  <div style="padding:16px; color:#aaa; text-align:center;">
    [ blocks go here ]
  </div>
  <div class="footer">
    <div>12 blocks</div>
    <div>last updated 2025-03-08</div>
  </div>
</body>
</html>
"""
preview(flex_nav, 230)

---
### Grid

Grid is **two-dimensional** — it defines rows and columns at the same time.

| Property | What it does |
|---|---|
| `display: grid` | Activates grid |
| `grid-template-columns` | Defines column widths |
| `grid-template-rows` | Defines row heights |
| `gap` | Space between cells |
| `grid-column: span N` | On children — span N columns |
| `grid-row: span N` | On children — span N rows |
| `fr` | Fractional unit — `1fr 2fr` means second column is twice as wide |
| `repeat(3, 1fr)` | Shorthand for `1fr 1fr 1fr` |
| `repeat(auto-fill, minmax(200px, 1fr))` | Responsive — as many columns as fit, each at least 200px |

#### Basic grid and spanning

In [ ]:
grid_basics = """
<!DOCTYPE html>
<html>
<head>
<style>
  body { margin: 16px; font-family: 'Times New Roman', Times, serif; font-size: 12px; }
  h3 { font-size: 11px; margin: 0 0 6px; color: #666; }

  .grid-a {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 8px;
    margin-bottom: 24px;
  }

  .cell {
    border: 1px solid #999;
    padding: 14px 10px;
    text-align: center;
  }

  .grid-b {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 8px;
  }
  .span-2 { grid-column: span 2; background: #eee; }
  .span-full { grid-column: 1 / -1; background: #ddd; }
</style>
</head>
<body>
  <h3>3-column grid</h3>
  <div class="grid-a">
    <div class="cell">1</div>
    <div class="cell">2</div>
    <div class="cell">3</div>
    <div class="cell">4</div>
    <div class="cell">5</div>
    <div class="cell">6</div>
  </div>

  <h3>with spanning</h3>
  <div class="grid-b">
    <div class="cell span-full">grid-column: 1 / -1 (full width)</div>
    <div class="cell span-2">span 2</div>
    <div class="cell">1</div>
    <div class="cell">1</div>
    <div class="cell span-2">span 2</div>
  </div>
</body>
</html>
"""
preview(grid_basics, 320)

**Try it:** Change `repeat(3, 1fr)` to `1fr 2fr 1fr` — the middle column gets twice the space.

#### Responsive grid with auto-fill

This is the pattern used in the Are.na page generator. The grid figures out how many columns fit automatically.

In [ ]:
grid_auto = """
<!DOCTYPE html>
<html>
<head>
<style>
  body { margin: 16px; font-family: 'Times New Roman', Times, serif; font-size: 12px; }
  h3 { font-size: 11px; margin: 0 0 6px; color: #666; }
  p.note { font-size: 11px; color: #888; margin: 0 0 10px; }

  .auto-grid {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(160px, 1fr));
    gap: 8px;
  }

  .block {
    border: 1px solid #999;
    padding: 12px;
  }
  .block .title { font-weight: bold; margin-bottom: 2px; }
  .block .type { color: #888; font-size: 10px; margin-bottom: 2px; }
  .block .date { color: #aaa; font-size: 10px; }
  .block .src { color: #aaa; font-size: 10px; word-break: break-all; }

  .block.no-image .thumb {
    width: 100%;
    height: 60px;
    background: #eee;
    margin-bottom: 6px;
    display: flex;
    align-items: center;
    justify-content: center;
    color: #bbb;
    font-size: 10px;
  }
</style>
</head>
<body>
  <h3>repeat(auto-fill, minmax(160px, 1fr))</h3>
  <p class="note">Columns adjust automatically. This is the layout from Step 4.</p>
  <div class="auto-grid">
    <div class="block">
      <div class="title">Storefront</div>
      <div class="type">Image</div>
      <div class="date">2025-01-12</div>
    </div>
    <div class="block">
      <div class="title">archived page</div>
      <div class="type">Link</div>
      <div class="src">geocities.com/...</div>
      <div class="date">2025-01-14</div>
    </div>
    <div class="block no-image">
      <div class="thumb">no image</div>
      <div class="title">Untitled</div>
      <div class="type">Attachment</div>
      <div class="date">2025-02-03</div>
    </div>
    <div class="block">
      <div class="title">"something about forgetting"</div>
      <div class="type">Text</div>
      <div class="date">2025-02-10</div>
    </div>
    <div class="block">
      <div class="title">Untitled</div>
      <div class="type">Image</div>
      <div class="date">2025-03-01</div>
    </div>
    <div class="block">
      <div class="title">404 page</div>
      <div class="type">Link</div>
      <div class="src">example.com/gone</div>
      <div class="date">2025-03-08</div>
    </div>
  </div>
</body>
</html>
"""
preview(grid_auto, 280)

**Try it:** Change `minmax(160px, 1fr)` to `minmax(280px, 1fr)`. Fewer columns, bigger cards.

#### Grid template areas

For page-level layout you can name regions and draw them out like a sketch.

In [ ]:
grid_areas = """
<!DOCTYPE html>
<html>
<head>
<style>
  body { margin: 0; font-family: 'Times New Roman', Times, serif; font-size: 12px; }

  .page {
    display: grid;
    grid-template-areas:
      "header  header"
      "sidebar main"
      "footer  footer";
    grid-template-columns: 180px 1fr;
    grid-template-rows: auto 1fr auto;
    min-height: 100vh;
  }

  .header  { grid-area: header;  padding: 12px 16px; border-bottom: 1px solid black; }
  .header h1 { font-size: 14px; margin: 0; }
  .sidebar { grid-area: sidebar; padding: 12px; border-right: 1px solid #ccc; }
  .main    { grid-area: main;    padding: 16px; }
  .footer  { grid-area: footer;  padding: 8px 16px; font-size: 10px; color: #999; border-top: 1px solid #ccc; }

  .sidebar ul { list-style: none; padding: 0; margin: 0; }
  .sidebar li { padding: 3px 0; border-bottom: 1px solid #eee; font-size: 11px; }

  .meta-table { width: 100%; border-collapse: collapse; margin-top: 10px; }
  .meta-table td { padding: 5px 8px; border-bottom: 1px solid #eee; font-size: 11px; vertical-align: top; }
  .meta-table td:first-child { font-weight: bold; color: #666; width: 100px; }
</style>
</head>
<body>
  <div class="page">
    <div class="header"><h1>channel title</h1></div>
    <div class="sidebar">
      <ul>
        <li>Storefront</li>
        <li>archived page</li>
        <li>Untitled</li>
        <li>"something about forgetting"</li>
        <li>Untitled</li>
        <li>404 page</li>
      </ul>
    </div>
    <div class="main">
      <b>Storefront</b>
      <table class="meta-table">
        <tr><td>type</td><td>Image</td></tr>
        <tr><td>connected</td><td>2025-01-12</td></tr>
        <tr><td>by</td><td>username</td></tr>
        <tr><td>source</td><td>—</td></tr>
        <tr><td>description</td><td><i>none</i></td></tr>
      </table>
    </div>
    <div class="footer">6 blocks · last updated 2025-03-08</div>
  </div>
</body>
</html>
"""
preview(grid_areas, 360)

**Try it:** Swap the sidebar to the right side:
```css
grid-template-areas:
  "header  header"
  "main    sidebar"
  "footer  footer";
grid-template-columns: 1fr 180px;
```

---
### Flexbox vs Grid — when to use which

| Use flexbox when... | Use grid when... |
|---|---|
| Laying out items in one direction | You need rows and columns |
| Content determines size (nav, tags) | Layout determines size (page, card grid) |
| Items should wrap in a row | You want precise 2D control |

In practice you combine them — grid for page structure, flexbox for content inside each cell.

---
---
# Part 3: Four Layout Options for Your Archive

Each layout below uses the same Are.na block data but arranges it differently. Pick one as a starting point, or combine ideas from several.

All use Times New Roman, black/white/gray only.

### Layout A: List

A table-like grid. Each row is a block. Every field gets a column.

Good for: metadata-heavy channels, emphasizing what's present and what's empty.

In [ ]:
layout_a = """
<!DOCTYPE html>
<html>
<head>
<style>
  body { margin: 0; font-family: 'Times New Roman', Times, serif; font-size: 12px; }

  .page { max-width: 860px; margin: 0 auto; padding: 30px 16px; }
  h1 { font-size: 14px; margin: 0 0 4px; }
  .sub { font-size: 11px; color: #888; margin-bottom: 20px; }

  .list {
    display: grid;
    grid-template-columns: 60px 1fr 60px 140px 80px;
    font-size: 11px;
  }

  .list-header {
    font-weight: bold;
    font-size: 10px;
    color: #888;
    padding: 4px 6px;
    border-bottom: 1px solid black;
  }

  .list-cell {
    padding: 6px;
    border-bottom: 1px solid #eee;
  }

  .list-cell.empty { color: #ccc; }

  .list-cell a { color: black; }
</style>
</head>
<body>
  <div class="page">
    <h1>channel title</h1>
    <div class="sub">6 blocks · from are.na</div>

    <div class="list">
      <div class="list-header">type</div>
      <div class="list-header">title</div>
      <div class="list-header">by</div>
      <div class="list-header">source</div>
      <div class="list-header">connected</div>

      <div class="list-cell">Image</div>
      <div class="list-cell">Storefront</div>
      <div class="list-cell">user1</div>
      <div class="list-cell empty">—</div>
      <div class="list-cell">2025-01-12</div>

      <div class="list-cell">Link</div>
      <div class="list-cell"><a href="#">archived page</a></div>
      <div class="list-cell">user1</div>
      <div class="list-cell" style="font-size:10px; word-break:break-all;">geocities.com/...</div>
      <div class="list-cell">2025-01-14</div>

      <div class="list-cell">Attachment</div>
      <div class="list-cell empty">Untitled</div>
      <div class="list-cell">user2</div>
      <div class="list-cell empty">—</div>
      <div class="list-cell">2025-02-03</div>

      <div class="list-cell">Text</div>
      <div class="list-cell">"something about forgetting"</div>
      <div class="list-cell">user1</div>
      <div class="list-cell empty">—</div>
      <div class="list-cell">2025-02-10</div>

      <div class="list-cell">Image</div>
      <div class="list-cell empty">Untitled</div>
      <div class="list-cell">user1</div>
      <div class="list-cell empty">—</div>
      <div class="list-cell">2025-03-01</div>

      <div class="list-cell">Link</div>
      <div class="list-cell"><a href="#">404 page</a></div>
      <div class="list-cell">user2</div>
      <div class="list-cell" style="font-size:10px; word-break:break-all;">example.com/gone</div>
      <div class="list-cell">2025-03-08</div>
    </div>
  </div>
</body>
</html>
"""
preview(layout_a, 310)

### Layout B: Card Grid

Responsive cards using `auto-fill`. Each card shows the block's image (or a placeholder), title, type, and date. This is closest to what the original notebook generates.

Good for: image-heavy channels, visual browsing.

In [ ]:
layout_b = """
<!DOCTYPE html>
<html>
<head>
<style>
  body { margin: 0; font-family: 'Times New Roman', Times, serif; font-size: 12px; background: white; }

  .page { max-width: 860px; margin: 0 auto; padding: 30px 16px; }
  h1 { font-size: 14px; margin: 0 0 4px; }
  .sub { font-size: 11px; color: #888; margin-bottom: 20px; }

  .card-grid {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(180px, 1fr));
    gap: 12px;
  }

  .card {
    border: 1px solid #999;
  }

  .card .thumb {
    width: 100%;
    aspect-ratio: 1 / 1;
    background: #eee;
    display: flex;
    align-items: center;
    justify-content: center;
    color: #aaa;
    font-size: 10px;
    border-bottom: 1px solid #ccc;
  }
  .card .thumb.has-image {
    background: #ddd;
    color: #999;
  }
  .card .thumb.text-block {
    background: black;
    color: #999;
    font-size: 9px;
    padding: 8px;
    text-align: left;
    align-items: flex-start;
    overflow: hidden;
    line-height: 1.3;
  }

  .card .info { padding: 8px 10px 10px; }
  .card .title { font-weight: bold; margin-bottom: 2px; font-size: 11px; }
  .card .title.empty { color: #aaa; font-weight: normal; }
  .card .meta { color: #888; font-size: 10px; }
  .card .src { color: #aaa; font-size: 9px; word-break: break-all; margin-top: 2px; }
</style>
</head>
<body>
  <div class="page">
    <h1>channel title</h1>
    <div class="sub">6 blocks · from are.na</div>

    <div class="card-grid">
      <div class="card">
        <div class="thumb has-image">[image]</div>
        <div class="info">
          <div class="title">Storefront</div>
          <div class="meta">Image · 2025-01-12</div>
        </div>
      </div>

      <div class="card">
        <div class="thumb has-image">[image]</div>
        <div class="info">
          <div class="title">archived page</div>
          <div class="meta">Link · 2025-01-14</div>
          <div class="src">geocities.com/...</div>
        </div>
      </div>

      <div class="card">
        <div class="thumb">no image</div>
        <div class="info">
          <div class="title empty">Untitled</div>
          <div class="meta">Attachment · 2025-02-03</div>
        </div>
      </div>

      <div class="card">
        <div class="thumb text-block">something about forgetting, the way a room looks different when you come back to it after...</div>
        <div class="info">
          <div class="title">"something about forgetting"</div>
          <div class="meta">Text · 2025-02-10</div>
        </div>
      </div>

      <div class="card">
        <div class="thumb has-image">[image]</div>
        <div class="info">
          <div class="title empty">Untitled</div>
          <div class="meta">Image · 2025-03-01</div>
        </div>
      </div>

      <div class="card">
        <div class="thumb">404</div>
        <div class="info">
          <div class="title">404 page</div>
          <div class="meta">Link · 2025-03-08</div>
          <div class="src">example.com/gone</div>
        </div>
      </div>
    </div>
  </div>
</body>
</html>
"""
preview(layout_b, 450)

### Layout C: Grouped by Type

Blocks sorted into sections by content type (Image, Link, Text, etc). Each section uses a flex column. Makes it easy to see what kinds of material you've collected.

Good for: mixed-media channels, showing the composition of your archive.

In [ ]:
layout_c = """
<!DOCTYPE html>
<html>
<head>
<style>
  body { margin: 0; font-family: 'Times New Roman', Times, serif; font-size: 12px; }

  .page { max-width: 620px; margin: 0 auto; padding: 30px 16px; }
  h1 { font-size: 14px; margin: 0 0 4px; }
  .sub { font-size: 11px; color: #888; margin-bottom: 24px; }

  .section {
    margin-bottom: 24px;
    padding-left: 16px;
    border-left: 2px solid black;
  }

  .section-label {
    font-size: 10px;
    color: #888;
    margin-bottom: 8px;
  }

  .section-items {
    display: flex;
    flex-direction: column;
    gap: 4px;
  }

  .entry {
    display: flex;
    justify-content: space-between;
    align-items: baseline;
    padding: 6px 8px;
    border: 1px solid #ddd;
  }
  .entry .name { font-size: 12px; }
  .entry .name.empty { color: #aaa; }
  .entry .date { font-size: 10px; color: #999; white-space: nowrap; margin-left: 12px; }
  .entry .src { font-size: 9px; color: #aaa; margin-left: 8px; }
</style>
</head>
<body>
  <div class="page">
    <h1>channel title</h1>
    <div class="sub">6 blocks · from are.na</div>

    <div class="section">
      <div class="section-label">Image (3)</div>
      <div class="section-items">
        <div class="entry">
          <span class="name">Storefront</span>
          <span class="date">2025-01-12</span>
        </div>
        <div class="entry">
          <span class="name empty">Untitled</span>
          <span class="date">2025-03-01</span>
        </div>
      </div>
    </div>

    <div class="section">
      <div class="section-label">Link (2)</div>
      <div class="section-items">
        <div class="entry">
          <span class="name">archived page</span>
          <span class="src">geocities.com</span>
          <span class="date">2025-01-14</span>
        </div>
        <div class="entry">
          <span class="name">404 page</span>
          <span class="src">example.com</span>
          <span class="date">2025-03-08</span>
        </div>
      </div>
    </div>

    <div class="section">
      <div class="section-label">Text (1)</div>
      <div class="section-items">
        <div class="entry">
          <span class="name">"something about forgetting"</span>
          <span class="date">2025-02-10</span>
        </div>
      </div>
    </div>

    <div class="section">
      <div class="section-label">Attachment (1)</div>
      <div class="section-items">
        <div class="entry">
          <span class="name empty">Untitled</span>
          <span class="date">2025-02-03</span>
        </div>
      </div>
    </div>

  </div>
</body>
</html>
"""
preview(layout_c, 440)

### Layout D: Mixed-Size Grid

An asymmetric grid where some blocks are bigger than others using `grid-column: span 2` and `grid-row: span 2`. Creates visual hierarchy.

Good for: highlighting certain blocks, editorial feel, breaking out of a uniform grid.

In [ ]:
layout_d = """
<!DOCTYPE html>
<html>
<head>
<style>
  body { margin: 0; font-family: 'Times New Roman', Times, serif; font-size: 12px; }

  .page { max-width: 860px; margin: 0 auto; padding: 30px 16px; }
  h1 { font-size: 14px; margin: 0 0 4px; }
  .sub { font-size: 11px; color: #888; margin-bottom: 20px; }

  .mixed-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    grid-auto-rows: 100px;
    gap: 6px;
  }

  .block {
    border: 1px solid #999;
    padding: 10px;
    display: flex;
    flex-direction: column;
    justify-content: space-between;
    overflow: hidden;
  }
  .block .title { font-weight: bold; font-size: 11px; }
  .block .title.empty { font-weight: normal; color: #aaa; }
  .block .meta { font-size: 9px; color: #888; margin-top: auto; }
  .block .desc { font-size: 10px; color: #666; margin-top: 4px; line-height: 1.3; }

  .block.big {
    grid-column: span 2;
    grid-row: span 2;
    background: black;
    color: white;
    border-color: black;
  }
  .block.big .meta { color: #666; }

  .block.tall { grid-row: span 2; }
  .block.wide { grid-column: span 2; }
</style>
</head>
<body>
  <div class="page">
    <h1>channel title</h1>
    <div class="sub">6 blocks · from are.na</div>

    <div class="mixed-grid">

      <div class="block big">
        <div class="title">Storefront</div>
        <div class="desc">the first image I saved. I keep coming back to the color of the awning.</div>
        <div class="meta">Image · 2025-01-12 · user1</div>
      </div>

      <div class="block">
        <div class="title">archived page</div>
        <div class="meta">Link · 2025-01-14</div>
      </div>

      <div class="block">
        <div class="title empty">Untitled</div>
        <div class="meta">Attachment · 2025-02-03</div>
      </div>

      <div class="block tall">
        <div class="title">"something about forgetting"</div>
        <div class="desc">something about forgetting, the way a room looks different when you come back to it after a long time away...</div>
        <div class="meta">Text · 2025-02-10</div>
      </div>

      <div class="block wide">
        <div class="title">404 page</div>
        <div class="desc">this link is dead now. it used to be a page about demolished buildings.</div>
        <div class="meta">Link · example.com/gone · 2025-03-08</div>
      </div>

      <div class="block">
        <div class="title empty">Untitled</div>
        <div class="meta">Image · 2025-03-01</div>
      </div>

    </div>
  </div>
</body>
</html>
"""
preview(layout_d, 430)

---
---
# Part 4: Generate Your Page

Now pick a layout and generate the HTML from your actual Are.na blocks.

Set `LAYOUT` to `'A'`, `'B'`, `'C'`, or `'D'`.

In [ ]:
# =============================================
#  CHANGE THIS to 'A', 'B', 'C', or 'D'
# =============================================
LAYOUT = 'B'

In [ ]:
from urllib.parse import urlparse

def get_block_fields(block):
    """Extract the usable fields from an Are.na block."""
    title = block.get('title') or block.get('generated_title') or ''
    cls = block.get('class', '')
    description = block.get('description') or ''
    content = block.get('content') or ''
    connected_at = (block.get('connected_at') or '')[:10]
    user = ''
    if block.get('user'):
        user = block['user'].get('username', '')
    source_url = ''
    source_domain = ''
    if block.get('source') and block['source'].get('url'):
        source_url = block['source']['url']
        try:
            source_domain = urlparse(source_url).hostname.replace('www.', '')
        except:
            source_domain = source_url[:40]
    img_url = ''
    img = block.get('image')
    if img and img.get('display') and img['display'].get('url'):
        img_url = img['display']['url']
    return {
        'title': title,
        'cls': cls,
        'description': description,
        'content': content,
        'connected_at': connected_at,
        'user': user,
        'source_url': source_url,
        'source_domain': source_domain,
        'img_url': img_url,
    }

def esc(s):
    """Minimal HTML escaping."""
    return s.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;').replace('"','&quot;')

print(f'Using {len(blocks)} blocks')

In [ ]:
# =============================================
#  LAYOUT A: List
# =============================================
def gen_layout_a(blocks):
    rows = ''
    for b in blocks:
        f = get_block_fields(b)
        title_display = esc(f['title']) if f['title'] else '<span style="color:#ccc;">Untitled</span>'
        if f['source_url']:
            title_display = f'<a href="{esc(f["source_url"])}" target="_blank">{title_display}</a>'
        src = esc(f['source_domain']) if f['source_domain'] else '<span style="color:#ccc;">&mdash;</span>'
        rows += f'''      <div class="list-cell">{esc(f['cls'])}</div>
      <div class="list-cell">{title_display}</div>
      <div class="list-cell">{esc(f['user'])}</div>
      <div class="list-cell" style="font-size:10px;word-break:break-all;">{src}</div>
      <div class="list-cell">{esc(f['connected_at'])}</div>
'''
    return f'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{esc(PAGE_TITLE)}</title>
  <style>
    * {{ margin: 0; padding: 0; box-sizing: border-box; }}
    body {{ font-family: 'Times New Roman', Times, serif; font-size: 12px; }}
    a {{ color: black; }}
    .page {{ max-width: 860px; margin: 0 auto; padding: 30px 16px; }}
    h1 {{ font-size: 14px; margin: 0 0 4px; }}
    .sub {{ font-size: 11px; color: #888; margin-bottom: 20px; }}
    .list {{
      display: grid;
      grid-template-columns: 60px 1fr 80px 140px 80px;
      font-size: 11px;
    }}
    .list-header {{
      font-weight: bold; font-size: 10px; color: #888;
      padding: 4px 6px; border-bottom: 1px solid black;
    }}
    .list-cell {{ padding: 6px; border-bottom: 1px solid #eee; }}
    @media (max-width: 600px) {{
      .list {{ grid-template-columns: 50px 1fr 60px; }}
      .list-header:nth-child(4), .list-header:nth-child(5),
      .list-cell:nth-child(5n+4), .list-cell:nth-child(5n+5) {{ display: none; }}
    }}
  </style>
</head>
<body>
  <div class="page">
    <h1>{esc(PAGE_TITLE)}</h1>
    <div class="sub">{len(blocks)} blocks &middot; from <a href="https://www.are.na/channel/{CHANNEL_SLUG}" target="_blank">are.na</a></div>
    <div class="list">
      <div class="list-header">type</div>
      <div class="list-header">title</div>
      <div class="list-header">by</div>
      <div class="list-header">source</div>
      <div class="list-header">connected</div>
{rows}    </div>
  </div>
</body>
</html>'''

print('Layout A generator ready')

In [ ]:
# =============================================
#  LAYOUT B: Card Grid
# =============================================
def gen_layout_b(blocks):
    cards = ''
    for b in blocks:
        f = get_block_fields(b)
        # thumbnail
        if f['img_url']:
            thumb = f'<img class="card-thumb" src="{esc(f["img_url"])}" alt="{esc(f["title"])}" loading="lazy" />'
        elif f['cls'] == 'Text' and f['content']:
            preview_text = esc(f['content'][:200])
            thumb = f'<div class="card-thumb text-thumb">{preview_text}</div>'
        else:
            thumb = f'<div class="card-thumb empty-thumb">{esc(f["cls"]) or "block"}</div>'
        # title
        title_cls = '' if f['title'] and f['title'] != 'Untitled' else ' empty'
        title_text = esc(f['title']) if f['title'] else 'Untitled'
        # source
        src_html = ''
        if f['source_domain']:
            src_html = f'<div class="card-src">{esc(f["source_domain"])}</div>'
        # description
        desc_html = ''
        if f['description']:
            short = esc(f['description'][:120])
            desc_html = f'<div class="card-desc">{short}</div>'

        cards += f'''    <div class="card">
      {thumb}
      <div class="card-info">
        <div class="card-title{title_cls}">{title_text}</div>
        <div class="card-meta">{esc(f['cls'])} &middot; {esc(f['connected_at'])}</div>
        {src_html}
        {desc_html}
      </div>
    </div>
'''
    return f'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{esc(PAGE_TITLE)}</title>
  <style>
    * {{ margin: 0; padding: 0; box-sizing: border-box; }}
    body {{ font-family: 'Times New Roman', Times, serif; font-size: 12px; }}
    a {{ color: black; }}
    .page {{ max-width: 860px; margin: 0 auto; padding: 30px 16px; }}
    h1 {{ font-size: 14px; margin: 0 0 4px; }}
    .sub {{ font-size: 11px; color: #888; margin-bottom: 20px; }}
    .card-grid {{
      display: grid;
      grid-template-columns: repeat(auto-fill, minmax(200px, 1fr));
      gap: 12px;
    }}
    .card {{ border: 1px solid #999; }}
    .card-thumb {{
      width: 100%; aspect-ratio: 1 / 1; object-fit: cover; display: block;
      border-bottom: 1px solid #ccc;
    }}
    .empty-thumb {{
      background: #eee; display: flex; align-items: center;
      justify-content: center; color: #aaa; font-size: 10px;
    }}
    .text-thumb {{
      background: black; color: #999; font-size: 9px; padding: 10px;
      line-height: 1.3; overflow: hidden;
    }}
    .card-info {{ padding: 8px 10px 10px; }}
    .card-title {{ font-weight: bold; font-size: 11px; margin-bottom: 2px; }}
    .card-title.empty {{ font-weight: normal; color: #aaa; }}
    .card-meta {{ color: #888; font-size: 10px; }}
    .card-src {{ color: #aaa; font-size: 9px; word-break: break-all; margin-top: 2px; }}
    .card-desc {{ color: #666; font-size: 10px; margin-top: 4px; font-style: italic; }}
    @media (max-width: 600px) {{
      .card-grid {{ grid-template-columns: 1fr 1fr; gap: 8px; }}
    }}
  </style>
</head>
<body>
  <div class="page">
    <h1>{esc(PAGE_TITLE)}</h1>
    <div class="sub">{len(blocks)} blocks &middot; from <a href="https://www.are.na/channel/{CHANNEL_SLUG}" target="_blank">are.na</a></div>
    <div class="card-grid">
{cards}    </div>
  </div>
</body>
</html>'''

print('Layout B generator ready')

In [ ]:
# =============================================
#  LAYOUT C: Grouped by Type
# =============================================
def gen_layout_c(blocks):
    from collections import OrderedDict
    groups = OrderedDict()
    for b in blocks:
        f = get_block_fields(b)
        cls = f['cls'] or 'Other'
        if cls not in groups:
            groups[cls] = []
        groups[cls].append(f)

    sections = ''
    for cls, items in groups.items():
        entries = ''
        for f in items:
            title_text = esc(f['title']) if f['title'] else '<span style="color:#aaa;">Untitled</span>'
            src = ''
            if f['source_domain']:
                src = f'<span class="entry-src">{esc(f["source_domain"])}</span>'
            entries += f'''        <div class="entry">
          <span class="entry-name">{title_text}</span>
          {src}
          <span class="entry-date">{esc(f['connected_at'])}</span>
        </div>
'''
        sections += f'''    <div class="section">
      <div class="section-label">{esc(cls)} ({len(items)})</div>
      <div class="section-items">
{entries}      </div>
    </div>
'''

    return f'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{esc(PAGE_TITLE)}</title>
  <style>
    * {{ margin: 0; padding: 0; box-sizing: border-box; }}
    body {{ font-family: 'Times New Roman', Times, serif; font-size: 12px; }}
    a {{ color: black; }}
    .page {{ max-width: 620px; margin: 0 auto; padding: 30px 16px; }}
    h1 {{ font-size: 14px; margin: 0 0 4px; }}
    .sub {{ font-size: 11px; color: #888; margin-bottom: 24px; }}
    .section {{ margin-bottom: 24px; padding-left: 16px; border-left: 2px solid black; }}
    .section-label {{ font-size: 10px; color: #888; margin-bottom: 8px; }}
    .section-items {{ display: flex; flex-direction: column; gap: 4px; }}
    .entry {{
      display: flex; justify-content: space-between; align-items: baseline;
      padding: 6px 8px; border: 1px solid #ddd;
    }}
    .entry-name {{ font-size: 12px; }}
    .entry-src {{ font-size: 9px; color: #aaa; margin-left: 8px; }}
    .entry-date {{ font-size: 10px; color: #999; white-space: nowrap; margin-left: 12px; }}
  </style>
</head>
<body>
  <div class="page">
    <h1>{esc(PAGE_TITLE)}</h1>
    <div class="sub">{len(blocks)} blocks &middot; from <a href="https://www.are.na/channel/{CHANNEL_SLUG}" target="_blank">are.na</a></div>
{sections}  </div>
</body>
</html>'''

print('Layout C generator ready')

In [ ]:
# =============================================
#  LAYOUT D: Mixed-Size Grid
# =============================================
def gen_layout_d(blocks):
    cards = ''
    for i, b in enumerate(blocks):
        f = get_block_fields(b)
        title_text = esc(f['title']) if f['title'] else '<span style="color:#aaa;">Untitled</span>'
        desc_html = ''
        if f['description']:
            desc_html = f'<div class="block-desc">{esc(f["description"][:160])}</div>'
        elif f['cls'] == 'Text' and f['content']:
            desc_html = f'<div class="block-desc">{esc(f["content"][:160])}</div>'
        meta_parts = [esc(f['cls'])]
        if f['source_domain']:
            meta_parts.append(esc(f['source_domain']))
        meta_parts.append(esc(f['connected_at']))
        if f['user']:
            meta_parts.append(esc(f['user']))
        meta = ' &middot; '.join(meta_parts)
        # make first block big, give text blocks extra height, make blocks with
        # long descriptions wide
        size_cls = ''
        if i == 0:
            size_cls = ' big'
        elif f['cls'] == 'Text' and len(f['content']) > 100:
            size_cls = ' tall'
        elif f['description'] and len(f['description']) > 80:
            size_cls = ' wide'

        cards += f'''    <div class="block{size_cls}">
      <div class="block-title">{title_text}</div>
      {desc_html}
      <div class="block-meta">{meta}</div>
    </div>
'''
    return f'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{esc(PAGE_TITLE)}</title>
  <style>
    * {{ margin: 0; padding: 0; box-sizing: border-box; }}
    body {{ font-family: 'Times New Roman', Times, serif; font-size: 12px; }}
    a {{ color: black; }}
    .page {{ max-width: 860px; margin: 0 auto; padding: 30px 16px; }}
    h1 {{ font-size: 14px; margin: 0 0 4px; }}
    .sub {{ font-size: 11px; color: #888; margin-bottom: 20px; }}
    .mixed-grid {{
      display: grid;
      grid-template-columns: repeat(4, 1fr);
      grid-auto-rows: 100px;
      gap: 6px;
    }}
    .block {{
      border: 1px solid #999; padding: 10px; overflow: hidden;
      display: flex; flex-direction: column; justify-content: space-between;
    }}
    .block-title {{ font-weight: bold; font-size: 11px; }}
    .block-desc {{ font-size: 10px; color: #666; margin-top: 4px; line-height: 1.3; }}
    .block-meta {{ font-size: 9px; color: #888; margin-top: auto; }}
    .block.big {{
      grid-column: span 2; grid-row: span 2;
      background: black; color: white; border-color: black;
    }}
    .block.big .block-meta {{ color: #666; }}
    .block.tall {{ grid-row: span 2; }}
    .block.wide {{ grid-column: span 2; }}
    @media (max-width: 600px) {{
      .mixed-grid {{ grid-template-columns: repeat(2, 1fr); grid-auto-rows: 80px; }}
    }}
  </style>
</head>
<body>
  <div class="page">
    <h1>{esc(PAGE_TITLE)}</h1>
    <div class="sub">{len(blocks)} blocks &middot; from <a href="https://www.are.na/channel/{CHANNEL_SLUG}" target="_blank">are.na</a></div>
    <div class="mixed-grid">
{cards}    </div>
  </div>
</body>
</html>'''

print('Layout D generator ready')

In [ ]:
# =============================================
#  Generate the page
# =============================================
generators = {'A': gen_layout_a, 'B': gen_layout_b, 'C': gen_layout_c, 'D': gen_layout_d}

if LAYOUT.upper() not in generators:
    print(f'Unknown layout: {LAYOUT}. Use A, B, C, or D.')
else:
    html = generators[LAYOUT.upper()](blocks)
    print(f'Layout {LAYOUT.upper()} generated: {len(html)} characters, {len(blocks)} blocks')

---
## Step 5: Save and download

In [ ]:
filename = f'{CHANNEL_SLUG}.html'
with open(filename, 'w', encoding='utf-8') as f:
    f.write(html)

print(f'Saved: {filename}')
print()

try:
    from google.colab import files
    files.download(filename)
    print('Download started.')
except ImportError:
    print(f'Not running in Colab. File saved to: {filename}')

---
## Step 6: Preview

In [ ]:
from IPython.display import HTML, display
display(HTML(f'<iframe srcdoc="{html.replace(chr(34), "&quot;")}" width="100%" height="600" style="border:1px solid #ccc;"></iframe>'))

---
---
# Part 5: Pagination — Controlling How Many Blocks Per Page

If your channel has a lot of blocks, you probably don't want to show them all at once. This section adds pagination to your generated page — you set how many blocks appear per page, and the page gets previous/next links to move between pages.

The approach: we split the blocks into chunks in Python, then generate a separate HTML file for each chunk with navigation links between them.

In [ ]:
# =============================================
#  Set how many blocks per page
# =============================================
BLOCKS_PER_PAGE = 6

The cell below splits your blocks into pages and generates one HTML file per page. Each file includes previous/next links at the bottom.

It uses whichever `LAYOUT` you set earlier.

In [ ]:
import math

def paginate(blocks, per_page):
    """Split blocks into a list of pages (each page is a list of blocks)."""
    pages = []
    for i in range(0, len(blocks), per_page):
        pages.append(blocks[i:i + per_page])
    return pages

def add_pagination_nav(html_content, page_num, total_pages, slug):
    """Insert pagination nav before the closing </body> tag."""
    if total_pages <= 1:
        return html_content

    # build prev/next links
    prev_link = ''
    next_link = ''
    if page_num > 1:
        prev_file = f'{slug}.html' if page_num == 2 else f'{slug}-{page_num - 1}.html'
        prev_link = f'<a href="{prev_file}">&larr; prev</a>'
    else:
        prev_link = '<span style="color:#ccc;">&larr; prev</span>'

    if page_num < total_pages:
        next_file = f'{slug}-{page_num + 1}.html'
        next_link = f'<a href="{next_file}">next &rarr;</a>'
    else:
        next_link = '<span style="color:#ccc;">next &rarr;</span>'

    nav_html = f'''
  <div style="max-width:860px; margin:20px auto; padding:12px 16px;
              display:flex; justify-content:space-between; align-items:center;
              border-top:1px solid #ccc; font-size:11px;
              font-family:'Times New Roman',Times,serif;">
    {prev_link}
    <span style="color:#888;">page {page_num} of {total_pages}</span>
    {next_link}
  </div>'''

    return html_content.replace('</body>', nav_html + '\n</body>')


# split into pages
pages = paginate(blocks, BLOCKS_PER_PAGE)
total_pages = len(pages)
print(f'{len(blocks)} blocks / {BLOCKS_PER_PAGE} per page = {total_pages} pages')

# generate and save each page
gen = generators[LAYOUT.upper()]
filenames = []

for i, page_blocks in enumerate(pages):
    page_num = i + 1
    page_html = gen(page_blocks)
    page_html = add_pagination_nav(page_html, page_num, total_pages, CHANNEL_SLUG)
    # first page uses the plain slug, rest get a number
    if page_num == 1:
        fname = f'{CHANNEL_SLUG}.html'
    else:
        fname = f'{CHANNEL_SLUG}-{page_num}.html'
    with open(fname, 'w', encoding='utf-8') as f:
        f.write(page_html)
    filenames.append(fname)
    print(f'  saved: {fname} ({len(page_blocks)} blocks)')

print()
print('Done. Download all files below.')

In [ ]:
# download all paginated files
try:
    from google.colab import files
    for fname in filenames:
        files.download(fname)
    print(f'Downloading {len(filenames)} files.')
except ImportError:
    print('Not in Colab. Files saved to current directory:')
    for fname in filenames:
        print(f'  {fname}')

Preview the first page:

In [ ]:
with open(filenames[0], 'r') as f:
    preview_html = f.read()
display(HTML(f'<iframe srcdoc="{preview_html.replace(chr(34), "&quot;")}" width="100%" height="600" style="border:1px solid #ccc;"></iframe>'))

---
---
# Part 6: Multiple Are.na Channels in One Archive

Your archive can pull from more than one Are.na channel. This section shows how to fetch blocks from multiple channels and combine them into a single page, with each channel labeled.

This is useful when your collection spans several channels, or when you want to bring together material from different sources into one archive interface.

In [ ]:
# =============================================
#  List your channels here
# =============================================
CHANNELS = [
    'relics-uikeqpbrexa',
    'dreamy-insecam',
]

MULTI_PAGE_TITLE = 'combined archive'

In [ ]:
import requests

all_channels = []  # list of {title, slug, blocks}

for slug in CHANNELS:
    url = f'https://api.are.na/v2/channels/{slug}?per=100'
    resp = requests.get(url)
    if resp.status_code != 200:
        print(f'Error fetching {slug}: {resp.status_code}')
        continue
    d = resp.json()
    ch_blocks = [b for b in d.get('contents', []) if b.get('class') != 'Channel']
    ch_title = d.get('title', slug)
    # tag each block with its source channel
    for b in ch_blocks:
        b['_channel_title'] = ch_title
        b['_channel_slug'] = slug
    all_channels.append({'title': ch_title, 'slug': slug, 'blocks': ch_blocks})
    print(f'{ch_title}: {len(ch_blocks)} blocks')

print(f'\nTotal: {sum(len(ch["blocks"]) for ch in all_channels)} blocks from {len(all_channels)} channels')

### Option 1: One page with sections per channel

Each channel gets its own labeled section. Uses a flex column layout with a border to separate them.

In [ ]:
def gen_multi_sections(channels, page_title):
    """Generate a page with a section for each channel."""
    sections = ''
    for ch in channels:
        entries = ''
        for b in ch['blocks']:
            f = get_block_fields(b)
            title_text = esc(f['title']) if f['title'] else '<span style="color:#aaa;">Untitled</span>'
            if f['source_url']:
                title_text = f'<a href="{esc(f["source_url"])}" target="_blank">{title_text}</a>'
            src = ''
            if f['source_domain']:
                src = f'<span class="entry-src">{esc(f["source_domain"])}</span>'
            entries += f'''        <div class="entry">
          <span class="entry-type">{esc(f['cls'])}</span>
          <span class="entry-name">{title_text}</span>
          {src}
          <span class="entry-date">{esc(f['connected_at'])}</span>
        </div>
'''
        sections += f'''    <div class="channel-section">
      <div class="channel-label">
        <a href="https://www.are.na/channel/{esc(ch['slug'])}" target="_blank">{esc(ch['title'])}</a>
        <span class="channel-count">{len(ch['blocks'])} blocks</span>
      </div>
      <div class="channel-items">
{entries}      </div>
    </div>
'''

    total = sum(len(ch['blocks']) for ch in channels)
    return f'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{esc(page_title)}</title>
  <style>
    * {{ margin: 0; padding: 0; box-sizing: border-box; }}
    body {{ font-family: 'Times New Roman', Times, serif; font-size: 12px; }}
    a {{ color: black; }}
    .page {{ max-width: 720px; margin: 0 auto; padding: 30px 16px; }}
    h1 {{ font-size: 14px; margin: 0 0 4px; }}
    .sub {{ font-size: 11px; color: #888; margin-bottom: 24px; }}

    .channel-section {{ margin-bottom: 28px; padding-left: 16px; border-left: 2px solid black; }}
    .channel-label {{
      font-size: 13px; font-weight: bold; margin-bottom: 10px;
      display: flex; justify-content: space-between; align-items: baseline;
    }}
    .channel-label a {{ text-decoration: none; }}
    .channel-label a:hover {{ text-decoration: underline; }}
    .channel-count {{ font-weight: normal; font-size: 10px; color: #888; }}
    .channel-items {{ display: flex; flex-direction: column; gap: 3px; }}
    .entry {{
      display: flex; align-items: baseline; gap: 8px;
      padding: 5px 8px; border: 1px solid #eee;
    }}
    .entry-type {{ font-size: 10px; color: #888; min-width: 55px; }}
    .entry-name {{ font-size: 12px; flex: 1; }}
    .entry-src {{ font-size: 9px; color: #aaa; }}
    .entry-date {{ font-size: 10px; color: #999; white-space: nowrap; }}
  </style>
</head>
<body>
  <div class="page">
    <h1>{esc(page_title)}</h1>
    <div class="sub">{total} blocks from {len(channels)} channels</div>
{sections}  </div>
</body>
</html>'''


multi_html = gen_multi_sections(all_channels, MULTI_PAGE_TITLE)

fname = 'combined-archive.html'
with open(fname, 'w', encoding='utf-8') as f:
    f.write(multi_html)
print(f'Saved: {fname}')

display(HTML(f'<iframe srcdoc="{multi_html.replace(chr(34), "&quot;")}" width="100%" height="600" style="border:1px solid #ccc;"></iframe>'))

### Option 2: All blocks mixed together in one grid

Combine all blocks from all channels into a single card grid. Each card shows which channel it came from.

In [ ]:
def gen_multi_mixed(channels, page_title):
    """Generate a single card grid with blocks from all channels mixed together."""
    # merge all blocks and sort by connected_at
    all_blocks = []
    for ch in channels:
        all_blocks.extend(ch['blocks'])
    all_blocks.sort(key=lambda b: b.get('connected_at', ''), reverse=True)

    cards = ''
    for b in all_blocks:
        f = get_block_fields(b)
        ch_title = b.get('_channel_title', '')
        # thumbnail
        if f['img_url']:
            thumb = f'<img class="card-thumb" src="{esc(f["img_url"])}" alt="{esc(f["title"])}" loading="lazy" />'
        elif f['cls'] == 'Text' and f['content']:
            thumb = f'<div class="card-thumb text-thumb">{esc(f["content"][:200])}</div>'
        else:
            thumb = f'<div class="card-thumb empty-thumb">{esc(f["cls"]) or "block"}</div>'
        title_cls = '' if f['title'] and f['title'] != 'Untitled' else ' empty'
        title_text = esc(f['title']) if f['title'] else 'Untitled'

        cards += f'''    <div class="card">
      {thumb}
      <div class="card-info">
        <div class="card-title{title_cls}">{title_text}</div>
        <div class="card-meta">{esc(f['cls'])} &middot; {esc(f['connected_at'])}</div>
        <div class="card-channel">{esc(ch_title)}</div>
      </div>
    </div>
'''

    return f'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{esc(page_title)}</title>
  <style>
    * {{ margin: 0; padding: 0; box-sizing: border-box; }}
    body {{ font-family: 'Times New Roman', Times, serif; font-size: 12px; }}
    a {{ color: black; }}
    .page {{ max-width: 860px; margin: 0 auto; padding: 30px 16px; }}
    h1 {{ font-size: 14px; margin: 0 0 4px; }}
    .sub {{ font-size: 11px; color: #888; margin-bottom: 20px; }}
    .card-grid {{
      display: grid;
      grid-template-columns: repeat(auto-fill, minmax(200px, 1fr));
      gap: 12px;
    }}
    .card {{ border: 1px solid #999; }}
    .card-thumb {{
      width: 100%; aspect-ratio: 1 / 1; object-fit: cover; display: block;
      border-bottom: 1px solid #ccc;
    }}
    .empty-thumb {{
      background: #eee; display: flex; align-items: center;
      justify-content: center; color: #aaa; font-size: 10px;
    }}
    .text-thumb {{
      background: black; color: #999; font-size: 9px; padding: 10px;
      line-height: 1.3; overflow: hidden;
    }}
    .card-info {{ padding: 8px 10px 10px; }}
    .card-title {{ font-weight: bold; font-size: 11px; margin-bottom: 2px; }}
    .card-title.empty {{ font-weight: normal; color: #aaa; }}
    .card-meta {{ color: #888; font-size: 10px; }}
    .card-channel {{ color: #aaa; font-size: 9px; margin-top: 3px; font-style: italic; }}
    @media (max-width: 600px) {{
      .card-grid {{ grid-template-columns: 1fr 1fr; gap: 8px; }}
    }}
  </style>
</head>
<body>
  <div class="page">
    <h1>{esc(page_title)}</h1>
    <div class="sub">{len(all_blocks)} blocks from {len(channels)} channels</div>
    <div class="card-grid">
{cards}    </div>
  </div>
</body>
</html>'''


mixed_html = gen_multi_mixed(all_channels, MULTI_PAGE_TITLE)

fname = 'combined-mixed.html'
with open(fname, 'w', encoding='utf-8') as f:
    f.write(mixed_html)
print(f'Saved: {fname}')

display(HTML(f'<iframe srcdoc="{mixed_html.replace(chr(34), "&quot;")}" width="100%" height="600" style="border:1px solid #ccc;"></iframe>'))

In [ ]:
# download combined files
try:
    from google.colab import files
    files.download('combined-archive.html')
    files.download('combined-mixed.html')
    print('Downloads started.')
except ImportError:
    print('Not in Colab. Files saved to current directory.')

---
---
## Reference

### Flexbox vs Grid cheat sheet

```
FLEXBOX (1D)                     GRID (2D)
------------------------------   ------------------------------
display: flex                    display: grid
flex-direction: row | column     grid-template-columns: ...
justify-content: ...             grid-template-rows: ...
align-items: ...                 grid-template-areas: ...
flex-wrap: wrap                  gap: ...
gap: ...
                                 on children:
on children:                     grid-column: span N
flex: grow shrink basis          grid-row: span N
align-self: ...                  grid-area: name
```

### Layout summary

| Layout | CSS used | Good for |
|---|---|---|
| A: List | grid with fixed column widths | metadata-heavy, tabular |
| B: Card Grid | `repeat(auto-fill, minmax())` | images, visual browsing |
| C: Grouped | flex columns inside sections | seeing composition by type |
| D: Mixed-Size | grid with `span 2` | editorial, hierarchy |

### Exercises

1. **Combine layouts:** Use Layout A for the index and grid-template-areas (from the teaching section) for individual block detail views.
2. **Add a header/footer:** Use the flexbox nav example from Part 2 and combine it with any layout.
3. **Show empty fields:** In your chosen layout, make blocks with no title, no description, or no source visually different — try lighter text, a dashed border, or a placeholder like `—`.
4. **Filter by type:** Add the filter bar from Part 2 and use JavaScript to show/hide blocks by their content type.

### Further reading
- [A Complete Guide to Flexbox](https://css-tricks.com/snippets/css/a-guide-to-flexbox/) — CSS-Tricks
- [A Complete Guide to Grid](https://css-tricks.com/snippets/css/complete-guide-grid/) — CSS-Tricks
- [Flexbox Froggy](https://flexboxfroggy.com/) — learn flexbox with a game
- [Grid Garden](https://cssgridgarden.com/) — learn grid with a game
- [Are.na API docs](https://dev.are.na/documentation)